# Preparação de tabelas para o Power BI

Este notebook usa os dados processados de emprego, população e educação para produzir a dimensão municipal e as tabelas de indicadores, atividades econômicas e cursos.

#### 1. Inspecionar a base municipal consolidada

In [1]:
import pandas as pd
from pathlib import Path

# Localiza os arquivos processados
pasta_processed = Path("../data/processed")

# Carrega a base municipal consolidada
indicadores = pd.read_parquet(
    pasta_processed / "indicadores_completos_municipais.parquet"
)

# Inspeciona sua estrutura
print("Dimensões:", indicadores.shape)
print("\nColunas:")
print(indicadores.columns.tolist())

print("\nMunicípios únicos:",
      indicadores["codigo_ibge"].nunique())

print("Códigos duplicados:",
      indicadores["codigo_ibge"].duplicated().sum())

Dimensões: (4441, 30)

Colunas:
['codigo_ibge', '2024', '2025', 'variacao_%', 'variacao_absoluta', '2026_parcial', 'admissoes', 'desligamentos', 'saldo', 'instituicoes_tecnicas', 'ofertas_cursos', 'cursos_distintos', 'matriculas_tecnicas', 'turmas_tecnicas', 'turmas_tecnicas_ead', 'percentual_ead', 'municipio', 'uf', 'tem_migracao', 'tem_emprego', 'tem_educacao', 'fontes_disponiveis', 'cursos_presenciais', 'ies_presenciais', 'matriculas_presenciais', 'matriculas_estrangeiros_presencial', 'matriculas_ead', 'ingressantes_ead', 'matriculas_estrangeiros_ead', 'tem_superior']

Municípios únicos: 4441
Códigos duplicados: 0


In [ ]:
# Atualiza as colunas de migração sem depender de uma nova execução do notebook 04.
migracao_atualizada = pd.read_parquet(
    pasta_processed / "migracao/indicadores_migracao_municipal.parquet"
)
colunas_migracao = [c for c in migracao_atualizada.columns if c not in ["codigo_ibge", "municipio", "uf"]]
indicadores = indicadores.drop(columns=colunas_migracao, errors="ignore").merge(
    migracao_atualizada.drop(columns=["municipio", "uf"]),
    on="codigo_ibge", how="outer", validate="one_to_one"
)

### 2. Integrar CAGED geral à base municipal

In [2]:
# Carrega os indicadores municipais do CAGED geral
emprego_geral = pd.read_parquet(
    pasta_processed / "trabalho/emprego_geral_municipal_2025.parquet"
)

# Renomeia as colunas para diferenciar emprego geral e migrante
emprego_geral = emprego_geral.rename(columns={
    "admissoes": "admissoes_gerais",
    "desligamentos": "desligamentos_gerais",
    "saldo": "saldo_geral"
})

# Integra os dados, preservando municípios das duas bases
indicadores = indicadores.merge(
    emprego_geral[
        ["codigo_ibge", "admissoes_gerais",
         "desligamentos_gerais", "saldo_geral"]
    ],
    on="codigo_ibge",
    how="outer",
    validate="one_to_one"
)

# Valida a integração
print("Dimensões:", indicadores.shape)
print("Municípios únicos:", indicadores["codigo_ibge"].nunique())
print("Códigos duplicados:", indicadores["codigo_ibge"].duplicated().sum())
print("Municípios com CAGED geral:", indicadores["admissoes_gerais"].notna().sum())

Dimensões: (5567, 33)
Municípios únicos: 5567
Códigos duplicados: 0
Municípios com CAGED geral: 5566


### 3. Integrar a população do IBGE 2026

Já trabalhamos com essa base no notebook 04_integracao_indicadores.ipynb. Se a variável populacao_2026 ainda estiver disponível naquele notebook, não precisamos repetir a limpeza.

In [3]:
# Carrega a população municipal já tratada
populacao = pd.read_parquet(
    pasta_processed / "populacao_municipal_2026.parquet"
)

# Integra a população aos indicadores municipais
indicadores = indicadores.merge(
    populacao,
    on="codigo_ibge",
    how="left",
    validate="one_to_one"
)

# Verifica o resultado
print("Dimensões:", indicadores.shape)
print("Populações ausentes:", indicadores["populacao_2026"].isna().sum())
print("Códigos duplicados:", indicadores["codigo_ibge"].duplicated().sum())

Dimensões: (5567, 34)
Populações ausentes: 0
Códigos duplicados: 0


### 4. Salvar a base municipal consolidada

In [4]:
# Cria a pasta para os indicadores integrados
pasta_integrados = pasta_processed / "integrados"
pasta_integrados.mkdir(parents=True, exist_ok=True)

# Salva a base municipal consolidada
indicadores.to_parquet(
    pasta_integrados / "indicadores_completos_municipais_v2.parquet",
    index=False
)

# Confirma a exportação
print("Arquivo salvo!")
print("Dimensões:", indicadores.shape)

Arquivo salvo!
Dimensões: (5567, 34)


### 4. Salvar a base municipal consolidada

In [5]:
# Seleciona os identificadores territoriais
dim_municipios = (
    indicadores[["codigo_ibge", "municipio", "uf"]]
    .drop_duplicates(subset="codigo_ibge")
    .sort_values("codigo_ibge")
    .reset_index(drop=True)
)

# Verifica a integridade da dimensão
print("Dimensões:", dim_municipios.shape)
print("Municípios sem nome:", dim_municipios["municipio"].isna().sum())
print("UFs ausentes:", dim_municipios["uf"].isna().sum())
print("Códigos duplicados:", dim_municipios["codigo_ibge"].duplicated().sum())

display(dim_municipios.head())

Dimensões: (5567, 3)
Municípios sem nome: 1126
UFs ausentes: 1126
Códigos duplicados: 0


,codigo_ibge,municipio,uf
0,1100015.0,ALTA FLORESTA D'OESTE,RO
1,1100023.0,ARIQUEMES,RO
2,1100031.0,NaN,NaN
3,1100049.0,CACOAL,RO
4,1100056.0,CEREJEIRAS,RO


#### 5. Recuperando os nomes e UFs dos municípios que não possuem população

In [6]:
# Carrega a referência oficial de municípios
municipios_ibge = pd.read_parquet(
    pasta_processed / "municipios_ibge.parquet"
)

# Inspeciona as colunas e os primeiros registros
print("Dimensões:", municipios_ibge.shape)
print("Colunas:", municipios_ibge.columns.tolist())

display(municipios_ibge.head())

Dimensões: (5571, 3)
Colunas: ['codigo_ibge', 'municipio', 'uf']


,codigo_ibge,municipio,uf
0,1100015,ALTA FLORESTA D'OESTE,RO
1,1100023,ARIQUEMES,RO
2,1100031,CABIXI,RO
3,1100049,CACOAL,RO
4,1100056,CEREJEIRAS,RO


In [7]:
# Preenche município e UF usando a referência oficial do IBGE
indicadores = (
    indicadores.set_index("codigo_ibge")
    .combine_first(
        municipios_ibge.set_index("codigo_ibge")[["municipio", "uf"]]
    )
    .reset_index()
)

# Recria a dimensão territorial com os dados atualizados
dim_municipios = indicadores[
    ["codigo_ibge", "municipio", "uf"]
].copy()

# Valida a cobertura territorial
print("Dimensões:", indicadores.shape)
print("Municípios sem nome:", indicadores["municipio"].isna().sum())
print("UFs ausentes:", indicadores["uf"].isna().sum())
print("Códigos duplicados:", indicadores["codigo_ibge"].duplicated().sum())

Dimensões: (5571, 34)
Municípios sem nome: 0
UFs ausentes: 0
Códigos duplicados: 0


#### 6. Atualizando os arquivos consolidados

In [8]:
# Garante que o código IBGE seja um inteiro de 7 dígitos
indicadores["codigo_ibge"] = indicadores["codigo_ibge"].astype("int64")
dim_municipios["codigo_ibge"] = dim_municipios["codigo_ibge"].astype("int64")

# Atualiza a base integrada com a cobertura territorial completa
indicadores.to_parquet(
    pasta_integrados / "indicadores_completos_municipais_v2.parquet",
    index=False
)

# Salva a dimensão territorial para o Power BI
dim_municipios.to_parquet(
    pasta_integrados / "dim_municipios.parquet",
    index=False
)

# Confirma os arquivos
print("Base integrada:", indicadores.shape)
print("Dimensão municípios:", dim_municipios.shape)
print("Arquivos atualizados!")

Base integrada: (5571, 34)
Dimensão municípios: (5571, 3)
Arquivos atualizados!


### 7. Criando a tabela de indicadores municipais

In [9]:
# Separa os indicadores da dimensão territorial
fato_indicadores = indicadores.drop(
    columns=[
        "municipio", "uf",
        "tem_migracao", "tem_emprego",
        "tem_educacao", "tem_superior",
        "fontes_disponiveis"
    ]
).copy()

# Renomeia colunas para identificar melhor as fontes
fato_indicadores = fato_indicadores.rename(columns={
    "2022": "registros_migratorios_2022",
    "2023": "registros_migratorios_2023",
    "2024": "registros_migratorios_2024",
    "2025": "registros_migratorios_2025",
    "2026_parcial": "registros_migratorios_2026_parcial",
    "variacao_%": "variacao_registros_pct",
    "variacao_absoluta": "variacao_registros_absoluta",
    "admissoes": "admissoes_migrantes",
    "desligamentos": "desligamentos_migrantes",
    "saldo": "saldo_migrantes"
})

# Confere a estrutura
print("Dimensões:", fato_indicadores.shape)
print("Códigos duplicados:",
      fato_indicadores["codigo_ibge"].duplicated().sum())
print("Colunas:", fato_indicadores.columns.tolist())

Dimensões: (5571, 27)
Códigos duplicados: 0
Colunas: ['codigo_ibge', 'registros_migratorios_2024', 'registros_migratorios_2025', 'variacao_registros_pct', 'variacao_registros_absoluta', 'registros_migratorios_2026_parcial', 'admissoes_migrantes', 'desligamentos_migrantes', 'saldo_migrantes', 'instituicoes_tecnicas', 'ofertas_cursos', 'cursos_distintos', 'matriculas_tecnicas', 'turmas_tecnicas', 'turmas_tecnicas_ead', 'percentual_ead', 'cursos_presenciais', 'ies_presenciais', 'matriculas_presenciais', 'matriculas_estrangeiros_presencial', 'matriculas_ead', 'ingressantes_ead', 'matriculas_estrangeiros_ead', 'admissoes_gerais', 'desligamentos_gerais', 'saldo_geral', 'populacao_2026']


### 8. Exportando para o Power BI

In [10]:
# Cria a pasta das tabelas destinadas ao Power BI
pasta_powerbi = pasta_processed / "powerbi"
pasta_powerbi.mkdir(parents=True, exist_ok=True)

# Salva a tabela em Parquet e CSV
fato_indicadores.to_parquet(
    pasta_powerbi / "fato_indicadores.parquet",
    index=False
)

fato_indicadores.to_csv(
    pasta_powerbi / "fato_indicadores.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

# Confirma a exportação
print("Arquivos exportados!")
print("Dimensões:", fato_indicadores.shape)

Arquivos exportados!
Dimensões: (5571, 27)


In [11]:
# Exporta a dimensão territorial para o Power BI
dim_municipios.to_csv(
    pasta_powerbi / "dim_municipios.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print("Dimensão de municípios exportada!")

Dimensão de municípios exportada!


#### 9. Preparar os dados CNAE

In [12]:
# Carrega os dados de emprego migrante por município e grupo CNAE
emprego_migrantes_cnae = pd.read_parquet(
    pasta_processed / "trabalho/emprego_por_municipio_grupo_2025.parquet"
)

# Carrega os dados de emprego geral por município e CNAE
emprego_geral_cnae = pd.read_parquet(
    pasta_processed / "trabalho/emprego_geral_municipio_cnae_2025.parquet"
)

# Inspeciona a estrutura antes do cruzamento
for nome, df in [
    ("Migrantes", emprego_migrantes_cnae),
    ("Geral", emprego_geral_cnae)
]:
    print(f"\n{nome}: {df.shape}")
    print("Colunas:", df.columns.tolist())
    display(df.head(3))


Migrantes: (29601, 8)
Colunas: ['codigo_ibge', 'municipio', 'uf', 'grupo', 'admissoes', 'desligamentos', 'saldo', 'descricao']


,codigo_ibge,municipio,uf,grupo,admissoes,desligamentos,saldo,descricao
0,1100023,ARIQUEMES,RO,1.5,1,0,1,Pecuária
1,1100023,ARIQUEMES,RO,10.6,0,1,-1,"Moagem, fabricação de produtos amiláceos e de ..."
2,1100023,ARIQUEMES,RO,10.9,2,0,2,Fabricação de outros produtos alimentícios



Geral: (588508, 7)
Colunas: ['uf', 'seção', 'subclasse', 'admissoes', 'desligamentos', 'saldo', 'codigo_ibge']


,uf,seção,subclasse,admissoes,desligamentos,saldo,codigo_ibge
0,11,A,111302,13,10,3,1100015
1,11,A,119909,0,1,-1,1100015
2,11,A,134200,1,0,1,1100015


### 10. Padronizando o CNAE geral

In [13]:
# Recupera os 7 dígitos da subclasse CNAE
codigo_subclasse = (
    emprego_geral_cnae["subclasse"]
    .astype("string")
    .str.zfill(7)
)

# Extrai o grupo CNAE no formato da tabela de migrantes
emprego_geral_cnae["grupo"] = (
    codigo_subclasse.str[:3].astype(int) / 10
)

# Agrega o emprego geral por município e grupo CNAE
emprego_geral_grupo = (
    emprego_geral_cnae
    .groupby(["codigo_ibge", "grupo"])
    .agg(
        admissoes_gerais=("admissoes", "sum"),
        desligamentos_gerais=("desligamentos", "sum"),
        saldo_geral=("saldo", "sum")
    )
    .reset_index()
)

# Valida os totais antes e depois da agregação
print("Dimensões:", emprego_geral_grupo.shape)
print("Admissões originais:", emprego_geral_cnae["admissoes"].sum())
print("Admissões agrupadas:", emprego_geral_grupo["admissoes_gerais"].sum())
print("Desligamentos originais:", emprego_geral_cnae["desligamentos"].sum())
print("Desligamentos agrupados:", emprego_geral_grupo["desligamentos_gerais"].sum())
print("Chaves duplicadas:",
      emprego_geral_grupo.duplicated(["codigo_ibge", "grupo"]).sum())

Dimensões: (301632, 5)
Admissões originais: 26318285
Admissões agrupadas: 26318285
Desligamentos originais: 25061582
Desligamentos agrupados: 25061582
Chaves duplicadas: 0


### 11. Integrando CAGED geral e migrantes por grupo CNAE

#### Comparar emprego geral e migrante por atividade

O cruzamento usa município e grupo CNAE. A junção externa preserva combinações exclusivas de cada base. Os totais e a cobertura conferem o resultado da integração.

In [14]:
# Padroniza os nomes dos indicadores migrantes
migrantes_grupo = emprego_migrantes_cnae.rename(columns={
    "admissoes": "admissoes_migrantes",
    "desligamentos": "desligamentos_migrantes",
    "saldo": "saldo_migrantes"
})

# Integra os dados pelo município e grupo CNAE
fato_emprego_cnae = emprego_geral_grupo.merge(
    migrantes_grupo[
        ["codigo_ibge", "grupo", "descricao",
         "admissoes_migrantes", "desligamentos_migrantes",
         "saldo_migrantes"]
    ],
    on=["codigo_ibge", "grupo"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

# Confere a integração
print("Dimensões:", fato_emprego_cnae.shape)
print("Chaves duplicadas:",
      fato_emprego_cnae.duplicated(["codigo_ibge", "grupo"]).sum())

print("\nCorrespondência:")
print(fato_emprego_cnae["_merge"].value_counts())

print("\nAdmissões migrantes:",
      fato_emprego_cnae["admissoes_migrantes"].sum())
print("Desligamentos migrantes:",
      fato_emprego_cnae["desligamentos_migrantes"].sum())

Dimensões: (301632, 10)
Chaves duplicadas: 0

Correspondência:
_merge
left_only     272031
both           29601
right_only         0
Name: count, dtype: int64

Admissões migrantes: 393415.0
Desligamentos migrantes: 315484.0


In [15]:
# Conta os grupos distintos e as descrições ausentes
print("Grupos distintos:", fato_emprego_cnae["grupo"].nunique())
print("Linhas sem descrição:", fato_emprego_cnae["descricao"].isna().sum())

# Verifica quais grupos não possuem descrição em nenhum município
grupos_descritos = set(
    fato_emprego_cnae.loc[
        fato_emprego_cnae["descricao"].notna(), "grupo"
    ]
)

grupos_sem_descricao = (
    set(fato_emprego_cnae["grupo"].dropna()) - grupos_descritos
)

print("Grupos sem nenhuma descrição:", len(grupos_sem_descricao))
print("Códigos:", sorted(grupos_sem_descricao))

Grupos distintos: 282
Linhas sem descrição: 272031
Grupos sem nenhuma descrição: 11
Códigos: [1.7, 5.0, 19.1, 26.8, 30.5, 51.3, 64.5, 80.3, 84.3, 99.0, 99.9]


### 12. Completar as descrições CNAE

In [16]:
# Cria um dicionário com as descrições já identificadas
dicionario_grupos = (
    emprego_migrantes_cnae
    .dropna(subset=["descricao"])
    .drop_duplicates("grupo")
    .set_index("grupo")["descricao"]
)

# Preenche as descrições ausentes usando o código do grupo
fato_emprego_cnae["descricao"] = (
    fato_emprego_cnae["descricao"].fillna(
        fato_emprego_cnae["grupo"].map(dicionario_grupos)
    )
)

# Identifica provisoriamente os grupos sem descrição
fato_emprego_cnae["descricao"] = fato_emprego_cnae["descricao"].fillna(
    "Grupo CNAE " + fato_emprego_cnae["grupo"].astype(str)
    + " — Sem descrição"
)

# Confere o resultado
print("Descrições ausentes:", fato_emprego_cnae["descricao"].isna().sum())

display(
    fato_emprego_cnae[
        fato_emprego_cnae["descricao"].str.contains("Sem descrição")
    ][["grupo", "descricao"]].drop_duplicates()
)

Descrições ausentes: 0


,grupo,descricao
1651,64.5,Grupo CNAE 64.5 — Sem descrição
1693,84.3,Grupo CNAE 84.3 — Sem descrição
4192,99.9,Grupo CNAE 99.9 — Sem descrição
5587,99.0,Grupo CNAE 99.0 — Sem descrição
9324,19.1,Grupo CNAE 19.1 — Sem descrição
17363,1.7,Grupo CNAE 1.7 — Sem descrição
32675,80.3,Grupo CNAE 80.3 — Sem descrição
55684,5.0,Grupo CNAE 5.0 — Sem descrição
67777,26.8,Grupo CNAE 26.8 — Sem descrição
125926,30.5,Grupo CNAE 30.5 — Sem descrição


### 13. Exportando a tabela CNAE para o Power BI

In [18]:
# Remove a coluna auxiliar utilizada na validação do merge
fato_emprego_cnae = fato_emprego_cnae.drop(columns="_merge")

# Salva a tabela CNAE para o Power BI
fato_emprego_cnae.to_parquet(
    pasta_powerbi / "fato_emprego_cnae.parquet",
    index=False
)

fato_emprego_cnae.to_csv(
    pasta_powerbi / "fato_emprego_cnae.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

# Confirma a exportação
print("Dimensões:", fato_emprego_cnae.shape)
print("Descrições ausentes:", fato_emprego_cnae["descricao"].isna().sum())
print("Arquivo exportado!")

Dimensões: (301632, 9)
Descrições ausentes: 0
Arquivo exportado!


### 14. Preparar a tabela de cursos técnicos

In [20]:
# Carrega a oferta de cursos técnicos por município
fato_cursos = pd.read_parquet(
    pasta_processed / "educacao/educacao_municipio_curso_2025.parquet"
)

# Renomeia as colunas para facilitar o uso no Power BI
fato_cursos = fato_cursos.rename(columns={
    "CO_CURSO_EDUC_PROFISSIONAL": "codigo_curso",
    "NO_CURSO_EDUC_PROFISSIONAL": "curso",
    "NO_AREA_CURSO_PROFISSIONAL": "area_formacao",
    "instituicoes": "instituicoes_curso",
    "matriculas": "matriculas_curso"
})

# Valida a estrutura e os totais
print("Dimensões:", fato_cursos.shape)
print("Chaves duplicadas:",
      fato_cursos.duplicated(["codigo_ibge", "codigo_curso"]).sum())
print("Matrículas:", fato_cursos["matriculas_curso"].sum())
print("Municípios:", fato_cursos["codigo_ibge"].nunique())
print("Áreas de formação:", fato_cursos["area_formacao"].nunique())

Dimensões: (18793, 6)
Chaves duplicadas: 0
Matrículas: 2490145
Municípios: 3385
Áreas de formação: 14


In [21]:
# Exporta a oferta de cursos técnicos para o Power BI
fato_cursos.to_parquet(
    pasta_powerbi / "fato_cursos.parquet",
    index=False
)

fato_cursos.to_csv(
    pasta_powerbi / "fato_cursos.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

# Confirma a exportação
print("Arquivo exportado!")
print("Dimensões:", fato_cursos.shape)

Arquivo exportado!
Dimensões: (18793, 6)


In [22]:
# Seleciona os indicadores de ensino superior presentes na base final
colunas_superior = [
    "cursos_presenciais",
    "ies_presenciais",
    "matriculas_presenciais",
    "matriculas_ead",
    "ingressantes_ead",
    "matriculas_estrangeiros_presencial",
    "matriculas_estrangeiros_ead"
]

# Verifica a cobertura e os totais nacionais
print("Municípios com dados de ensino superior:")
print(fato_indicadores[colunas_superior].notna().any(axis=1).sum())

print("\nTotais dos indicadores:")
display(fato_indicadores[colunas_superior].sum())

Municípios com dados de ensino superior:
3551

Totais dos indicadores:


cursos_presenciais                      34479.0
ies_presenciais                          3793.0
matriculas_presenciais                5037875.0
matriculas_ead                        5186811.0
ingressantes_ead                      3346076.0
matriculas_estrangeiros_presencial      15981.0
matriculas_estrangeiros_ead              6939.0
dtype: float64

## 15. Migração 2022–2026, países e cadastro ativo

Execute antes os notebooks **01 e 08**. A exportação mantém os fatos de registros anuais separados do cadastro por situação e do recorte ativo. A dimensão de países usa chaves textuais do projeto, não códigos ISO; inclui categorias não informadas e históricas.

Relacione `dim_paises[pais_id]` à chave de país correspondente em cada fato; `dim_periodos_migracao[ano_registro]` somente aos fatos anuais; `dim_ufs[uf]` aos fatos estaduais; `dim_municipios[codigo_ibge]` aos fatos municipais. Use relações 1:N, com filtro da dimensão para o fato. Não relacione fatos entre si nem some níveis de agregação. Os percentuais exportados têm denominador fixo na origem; use razões de somas em medidas dinâmicas.

O campo `pais_fonte` do cadastro não comprova país de nascimento. `snapshot_id` não representa uma data de referência conhecida. Não compare o total anual de registros ao cadastro ativo como se ambos fossem população residente.

In [ ]:
import sys
RAIZ = Path.cwd() if (Path.cwd() / "src/migracao.py").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from src.migracao import exportar_powerbi_migracao

manifesto_migracao = exportar_powerbi_migracao(RAIZ)
display(manifesto_migracao)
print("Nove fatos e três dimensões de migração exportados em CSV e Parquet.")